In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
PROJECT_ROOT = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair"

WORKING_PATH = os.path.join(PROJECT_ROOT, "data", "working", "df_working.csv")
CLEANED_PATH = os.path.join(PROJECT_ROOT, "data", "cleaned", "df_cleaned.csv")
PLAN_PATH = os.path.join(PROJECT_ROOT, "reports", "repair_plan", "repair_plan.csv")

df_working = pd.read_csv(WORKING_PATH)
df_cleaned = pd.read_csv(CLEANED_PATH)
repair_plan = pd.read_csv(PLAN_PATH)

# normalize column names
df_working.columns = df_working.columns.str.strip().str.lower()
df_cleaned.columns = df_cleaned.columns.str.strip().str.lower()
repair_plan.columns = repair_plan.columns.str.strip().str.lower()

In [3]:
if df_working.empty:
    raise ValueError("df_working is empty")

if df_cleaned.empty:
    raise ValueError("df_cleaned is empty")

if repair_plan.empty:
    raise ValueError("repair_plan is empty")

In [4]:
ACTION_TO_CHANGE_TYPE = {
    "impute_missing": "value_imputed",
    "correct_invalid": "value_corrected",
    "cap_outliers": "value_capped",
    "evaluate_high_cardinality": "no_change",
    "mark_as_identifier": "no_change",
    "resolve_semantic_mismatch": "no_change",
    "flag_for_review": "no_change"
}

In [5]:
print("Missing in df_working:")
print(df_working.isna().sum().sort_values(ascending=False).head(5))

print("\nMissing in df_cleaned:")
print(df_cleaned.isna().sum().sort_values(ascending=False).head(5))

Missing in df_working:
occupation        2809
workclass         2799
native-country     857
fnlwgt               0
education            0
dtype: int64

Missing in df_cleaned:
age                0
workclass          0
fnlwgt             0
education          0
educational-num    0
dtype: int64


In [6]:
audit_records = []

In [7]:
for _, row in repair_plan.iterrows():

    col = str(row["column_name"]).strip().lower()
    action = row["recommended_action"]

    if action not in ACTION_TO_CHANGE_TYPE:
        raise ValueError(f"Unsupported action: {action}")

    if col not in df_working.columns:
        raise ValueError(f"Column missing in df_working: {col}")

    if col not in df_cleaned.columns:
        raise ValueError(f"Column missing in df_cleaned: {col}")

    working_col = df_working[col].reset_index(drop=True)
    cleaned_col = df_cleaned[col].reset_index(drop=True)

    change_type = ACTION_TO_CHANGE_TYPE[action]

    # =========================
    # TRUSTED CHANGE DETECTION
    # =========================

    if change_type == "value_imputed":

        missing_before = working_col.isna().sum()
        missing_after = cleaned_col.isna().sum()

        was_modified = missing_after < missing_before
        filled = missing_before - missing_after

        details = f"{filled} values imputed"

    else:
        # fallback comparison (safe for non-imputation)
        working_str = working_col.astype("string")
        cleaned_str = cleaned_col.astype("string")

        equal_mask = (
            (working_str == cleaned_str) |
            (working_col.isna() & cleaned_col.isna())
        )

        was_modified = not equal_mask.all()

        if change_type == "value_corrected":
            changes = (~equal_mask).sum()
            details = f"{changes} values corrected"

        elif change_type == "value_capped":
            changes = (~equal_mask).sum()
            details = f"{changes} values capped"

        else:
            details = "no transformation applied"

    audit_records.append({
        "column_name": col,
        "action_applied": action,
        "was_modified": bool(was_modified),
        "change_type": change_type,
        "details": details
    })

In [8]:
audit_log = pd.DataFrame(audit_records)

In [9]:
required_columns = [
    "column_name",
    "action_applied",
    "was_modified",
    "change_type",
    "details"
]

if audit_log.empty:
    raise ValueError("audit_log is empty")

if list(audit_log.columns) != required_columns:
    raise ValueError("audit_log schema mismatch")

if audit_log.isnull().any().any():
    raise ValueError("audit_log contains null values")

In [10]:
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "reports", "audit_logs")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "audit_log.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

audit_log.to_csv(OUTPUT_PATH, index=False)

print("Audit log saved to:", OUTPUT_PATH)

Audit log saved to: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/audit_logs/audit_log.csv
